In [ ]:
# used the instructions of https://www.sparkcodehub.com/pyspark/use-cases/recommendation-systems
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col

# SparkSession (like the wpo)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AnimeRecommender") \
    .config("spark.ui.port", "4040") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.extraJavaOptions", "-Duser.name=admin") \
    .config("spark.driver.extraJavaOptions", "-Dsun.security.auth.login.config=C:/dev/null") \
    .getOrCreate()

In [11]:
import os

# This tells Python to get the full path from the root of your hard drive
reviews_path = os.path.abspath("../data/preprocessed/reviews.csv")

#pre processed data
reviews_data = spark.read.csv(reviews_path, header=True, inferSchema=True)
anime_data = spark.read.csv("../data/preprocessed/animes.csv", header=True, inferSchema=True)

# The profiles need to be represented in numbers.
# Tried using cast like the intstructions but didn't work due to 
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col("user_id").cast("integer"),
    col("anime_uid").cast("integer"),
    col("score").cast("float")
)

final_data.show(7)

UnsupportedOperationException: getSubject is supported only if a security manager is allowed

In [ ]:
## proof cast won't work
# test_data = [("---SnowFlake---",), ("--EYEPATCH--",), ("123",)]
# test_df = spark.createDataFrame(test_data, ["profile"])
# result_df = test_df.withColumn("user_id_test", col("profile").cast("integer"))
# result_df.show()

{"ts": "2026-05-30 14:47:02.311", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '---SnowFlake---' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 5 in cell [2]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o57.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '---SnowFlake---' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 5 in cell [2]\n\r\n\tat org.apache.spark.sql.errors

NumberFormatException: [CAST_INVALID_INPUT] The value '---SnowFlake---' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 5 in cell [2]


In [ ]:
from pyspark.ml.recommendation import ALS

# Initialize ALS model
als = ALS(
    maxIter=10,              # Number of iterations
    regParam=0.1,            # Regularization parameter
    rank=10,                 # Number of latent factors
    userCol="user_id",       # User column
    itemCol="anime_id",      # Item column
    ratingCol="score",       # Rating column
    coldStartStrategy="drop" # Handle cold-start issues
)

# Train the model
model = als.fit(final_data)

In [ ]:
user_recs.limit(3).show(truncate=False)
user_recs.show(truncate=False)